In [11]:
#!/usr/bin/env python3
import os
import sys
import re
import json
import difflib
from typing import Optional, Tuple, List

import pandas as pd
import numpy as np
from pandas.tseries.offsets import MonthEnd, YearEnd


def get_period_columns(df: pd.DataFrame) -> pd.DataFrame:
    '''function for finding the period columns of a dataframe'''
    return

def data_quality_checks(df: pd.DataFrame) -> pd.DataFrame:
    # Check for missing values
    missing_values = df.isnull().sum()
    print("Missing values in each column:\n", missing_values)

    # Check for duplicates
    duplicates = df.duplicated().sum()
    print(f"Number of duplicate rows: {duplicates}")

    # Check data types
    print("Data types:\n", df.dtypes)

    # Additional checks can be added here

    return df

def get_monthly_income(df: pd.DataFrame) -> pd.DataFrame:
    # Assuming df has a column 'median_household_income' which is annual income
    # This function will convert it to monthly income so we can compare to rent in an understandble number
    if 'median_renters_income' not in df.columns:
        raise ValueError("DataFrame must contain 'median_renters_income' column")

    # Convert annual income to monthly income
    df['median_monthly_rent_income'] = df['median_renters_income'] / 12

    return df

def calculate_hai(df: pd.DataFrame) -> pd.DataFrame:
    # Assuming df has columns 'median_household_income' and 'median_home_price'
    if 'median_household_income' not in df.columns or 'median_home_value' not in df.columns:
        raise ValueError("DataFrame must contain 'median_household_income' and 'median_home_value' columns")

    # Calculate HAI
    df['HAI'] = (df['median_home_value'] / df['median_household_income'])

    # Handle potential division by zero or NaN values
    df['HAI'].replace([np.inf, -np.inf], np.nan, inplace=True)

    return df


def calculate_rai(df: pd.DataFrame) -> pd.DataFrame:
    # Assuming df has columns 'median_gross_rent' and 'median_home_price'
    if 'median_gross_rent' not in df.columns or 'median_monthly_rent_income' not in df.columns:
        raise ValueError("DataFrame must contain 'median_gross_rent' and 'median_home_value' columns")

    # Calculate RAI
    df['RAI'] = (df['median_monthly_rent_income'] / df['median_gross_rent'])

    # Handle potential division by zero or NaN values
    df['RAI'].replace([np.inf, -np.inf], np.nan, inplace=True)

    return df

def calculate_indexed_hai(df: pd.DataFrame) -> pd.DataFrame:
    df['HAI_Index'] = (df['median_household_income_indexed'] / df['hpi_value_indexed']) * 100
    
    # Handle potential division by zero or NaN values
    df['HAI_Index'].replace([np.inf, -np.inf], np.nan, inplace=True)

    return df

    
def index_variable(df: pd.DataFrame, variable_to_index:str, groups_column: Optional[str]) -> pd.DataFrame:
    ''' This function will index a variable based on a start year 
    INPUTS:
    
    variable_to_index - column name of value to index
    start_year - year to start the index
    
    '''
    if df[variable_to_index].dtypes not in [int,float]:
        raise ValueError(f'{variable_to_index} not a numeric')
    
    year_col = 'year' # TODO: Change to find the period column function when I need this functionality
    if "Year" in df.columns.tolist():   
        year_col = "Year"

    df['base_value'] = df.sort_values(year_col).groupby(groups_column)[variable_to_index].transform("first")
    df[f'{variable_to_index}_indexed'] = df[variable_to_index] / df['base_value']
    return df    


In [6]:
import pandas as pd
import os
from pathlib import Path
ROOT = Path.cwd().parent
PATHS = {
    "input_csv": os.path.join(ROOT, "data",'processed',"income_hpi_home_rent_at_county.parquet"),
    "shapefiles_dir": os.path.join(ROOT, "shapefiles"),
    "processed_dir": os.path.join(ROOT, "data", "processed"),
    "geo_dir": os.path.join(ROOT, "data", "geo"),
    "quality_dir": os.path.join(ROOT, "data", "quality"),
    "fig_maps_dir": os.path.join(ROOT, "figures", "maps"),
}

In [16]:
df = pd.read_parquet(PATHS["input_csv"])

In [17]:
df.shape

(1548299, 16)

In [14]:
df2 = get_monthly_income(df)
df3 = calculate_rai(df)
df4 = calculate_hai(df)

In [18]:
df4.columns

Index(['county_fips_full', 'year', 'median_household_income',
       'median_renters_income', 'income_change', 'county_name',
       'state_fips_home', 'county_fips_home', 'median_home_value',
       'source_home', 'county_name_home', 'state_fips_rent',
       'county_fips_rent', 'median_gross_rent', 'source_rent',
       'county_name_rent', 'median_monthly_rent_income', 'RAI', 'HAI'],
      dtype='object')